# AL-DIDAH — Anime Reverse Engineering & Production Pipeline
**Colab GPU notebook | GitHub Release artifacts | Two reference characters**

This notebook is designed as a restartable production pipeline. It downloads the source video and two character references, preserves the opening logo as a separately identified intro segment, analyzes scenes/motion/audio, prepares high-resolution character plates, creates production manifests and QA contact sheets, then publishes a versioned GitHub Release.

**Important:** a notebook cannot guarantee zero runtime failures or create a finished broadcast-quality anime episode from two still images. It provides validated analysis, character asset preparation, shot/motion data, and reproducible production handoff artifacts. Deep character animation/rendering remains a separate stage.

## 0. Runtime and inputs
Use **Runtime → Change runtime type → T4 GPU** (or better). Run cells top-to-bottom. Set the two reference image file IDs below if they differ. The supplied Drive folder ID is tried first; if it is inaccessible, the notebook stops with a clear diagnostic rather than silently selecting unrelated media.

In [ ]:
# Runtime configuration — edit only if the input locations change
PROJECT = "AL-DIDAH"
GITHUB_OWNER = "h776117073-eng"
GITHUB_REPO = "AI-Creative-Studio"
GITHUB_RELEASE_TAG = "al-didah-production-v1"
GITHUB_RELEASE_NAME = "AL-DIDAH Production Run v1"

# User-supplied Drive folder link (contains the source material)
DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1D5EaANB1ycUQ5SHJcjb6PjdS6UE372aV"
DRIVE_FOLDER_ID = "1D5EaANB1ycUQ5SHJcjb6PjdS6UE372aV"

# Previously identified reference image IDs in the connected project context.
# If these are not the two intended images, replace them with their Drive file IDs.
CHARACTER_IMAGE_IDS = [
    "1uY_3X4RZceENv7ns6BV0PGVVzcVlyiaO",
    "1snzt4jR2YiPoebpRjKa8sCqxAmW1KgtE"
]
# Known video source previously documented for this project; used only as fallback
# if the supplied Drive folder does not expose an MP4.
FALLBACK_VIDEO_URL = "https://youtu.be/d_qGO4GrbQM"

# Keep the opening logo in the analysis. It is classified separately; never auto-trim it.
PRESERVE_OPENING_LOGO = True
MAX_ANALYSIS_FPS = 6
MAX_LONG_SIDE = 1280
SEED = 42

from pathlib import Path
import os, json, time, hashlib, shutil, subprocess, sys, textwrap, glob
ROOT = Path("/content/al_didah")
INPUT = ROOT/"inputs"; WORK = ROOT/"work"; OUT = ROOT/"outputs"; ASSETS = OUT/"character_assets"
for p in (INPUT, WORK, OUT, ASSETS): p.mkdir(parents=True, exist_ok=True)
print("Workspace:", ROOT)

## 1. Install pinned, compatible dependencies
The notebook uses conservative versions and verifies imports after installation. Optional models are downloaded only when their stage is reached.

In [ ]:
%pip -q install "numpy<2" "opencv-python-headless==4.10.0.84" "pillow>=10,<12" "pandas>=2,<3" "matplotlib>=3.8,<4" "scenedetect[opencv]>=0.6.4" "ultralytics>=8.3,<9" "gdown>=5.2,<6" "yt-dlp>=2025.1" "rembg[cpu]>=2.0.60" "onnxruntime>=1.19" "tqdm>=4.66" "requests>=2.32"
import cv2, numpy as np, pandas as pd, PIL, requests
from PIL import Image, ImageOps, ImageFilter
print("OpenCV", cv2.__version__, "| NumPy", np.__version__, "| GPU:", subprocess.getoutput("nvidia-smi --query-gpu=name --format=csv,noheader"))

## 2. Fetch source media — fail safely
The Drive folder supplied in the request was not visible to the connected Drive integration during setup. This cell retries from inside Colab using the public/shareable folder URL. For successful access, set the folder or individual files to **Anyone with the link → Viewer**. No permissions are modified by this notebook.

In [ ]:
import re
def run(cmd, check=True):
    print("+", " ".join(map(str,cmd))[:240])
    return subprocess.run(cmd, check=check, text=True, capture_output=True)

# Try folder download. gdown can list/download publicly shared Drive folders.
folder_dir = INPUT/"drive_folder"; folder_dir.mkdir(exist_ok=True)
folder_ok = False
try:
    r = run(["gdown","--folder",DRIVE_FOLDER_URL,"-O",str(folder_dir),"--remaining-ok"], check=False)
    folder_ok = any(folder_dir.rglob("*.mp4")) or any(folder_dir.rglob("*.mkv")) or any(folder_dir.rglob("*.mov"))
    print("Drive folder fetch:", "media found" if folder_ok else (r.stderr[-1200:] if r.stderr else "no video found"))
except Exception as e:
    print("Drive folder fetch diagnostic:", repr(e))

# Fetch the two image IDs independently; public link permission is required.
import gdown
image_paths=[]
for idx, fid in enumerate(CHARACTER_IMAGE_IDS, 1):
    dest=INPUT/f"character_{idx}"
    dest.mkdir(exist_ok=True)
    try:
        url=f"https://drive.google.com/uc?id={fid}"
        r=run(["gdown",url,"-O",str(dest/"reference")],check=False)
        candidates=[p for p in dest.iterdir() if p.is_file() and p.stat().st_size>1024]
        if candidates: image_paths.append(candidates[0])
        else: print(f"Character image {idx} not fetched; verify its file ID and Viewer sharing.")
    except Exception as e: print(f"Character image {idx} error:",repr(e))

# Locate video in downloaded folder, prioritizing MP4 and largest file.
videos=[]
for ext in ("*.mp4","*.mkv","*.mov","*.webm"):
    videos += list(folder_dir.rglob(ext))
videos=sorted(videos,key=lambda p:(p.suffix.lower()==".mp4",p.stat().st_size),reverse=True)
video_path=videos[0] if videos else None
if video_path is None:
    print("No video found in the supplied Drive folder. Trying documented YouTube fallback.")
    try:
        run(["yt-dlp","-f","bv*[height<=1080]+ba/b[height<=1080]","--merge-output-format","mp4","-o",str(INPUT/"source.%(ext)s"),FALLBACK_VIDEO_URL],check=True)
        found=list(INPUT.glob("source.*"))
        video_path=found[0] if found else None
    except Exception as e: print("Fallback download failed:",repr(e))
if video_path is None or not video_path.exists():
    raise RuntimeError("SOURCE VIDEO MISSING. Share the Drive folder/video as Anyone with link → Viewer, or set a direct downloadable video URL.")
if len(image_paths)!=2:
    raise RuntimeError(f"Expected 2 reference images, fetched {len(image_paths)}. Verify CHARACTER_IMAGE_IDS and set each image to Anyone with link → Viewer.")
print("Video:",video_path, "bytes:",video_path.stat().st_size)
print("Character refs:",image_paths)

## 3. Input integrity, video metadata, and opening-logo handling
The first frames are explicitly preserved. The opening is recorded as a candidate intro segment, not removed. This prevents accidental loss of the logo/title card before the episode.

In [ ]:
import cv2, json, numpy as np
cap=cv2.VideoCapture(str(video_path))
if not cap.isOpened(): raise RuntimeError("OpenCV cannot open the downloaded source video.")
fps=float(cap.get(cv2.CAP_PROP_FPS) or 0)
frame_count=int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration=frame_count/fps if fps>0 else 0
if fps<=0 or frame_count<=0 or width<=0 or height<=0: raise RuntimeError("Invalid video metadata; file may be corrupt or incomplete.")
meta={"path":str(video_path),"fps":fps,"frame_count":frame_count,"width":width,"height":height,"duration_seconds":duration,"fourcc":int(cap.get(cv2.CAP_PROP_FOURCC))}
cap.release()
(OUT/"video_metadata.json").write_text(json.dumps(meta,indent=2),encoding="utf-8")
print(json.dumps(meta,indent=2))
# Opening frame strip for manual inspection
from IPython.display import display
from PIL import Image, ImageDraw
cap=cv2.VideoCapture(str(video_path)); samples=[]
for sec in [0,1,2,3,5,8,12,18]:
    cap.set(cv2.CAP_PROP_POS_MSEC,sec*1000); ok,fr=cap.read()
    if ok:
        fr=cv2.cvtColor(fr,cv2.COLOR_BGR2RGB); im=Image.fromarray(fr); im.thumbnail((320,180)); samples.append((sec,im.copy()))
cap.release()
sheet=Image.new("RGB",(640,((len(samples)+1)//2)*210),"white"); d=ImageDraw.Draw(sheet)
for i,(sec,im) in enumerate(samples):
    x=(i%2)*320; y=(i//2)*210; sheet.paste(im,(x,y+22)); d.text((x+5,y+4),f"{sec}s",fill="black")
display(sheet)
(OUT/"opening_logo_review.txt").write_text("Opening frames preserved. Inspect the contact sheet above and mark intro boundary manually if required. No automatic trim applied.\n",encoding="utf-8")

## 4. Scene segmentation and sampled frame extraction
Scene boundaries are detected with PySceneDetect. Analysis sampling is capped to control GPU/CPU cost; original timestamps and frame indices are retained.

In [ ]:
from scenedetect import open_video, SceneManager
from scenedetect.detectors import ContentDetector
v=open_video(str(video_path))
sm=SceneManager(); sm.add_detector(ContentDetector(threshold=27.0,min_scene_len=max(12,int(fps*0.5))))
sm.detect_scenes(v)
scene_list=sm.get_scene_list()
scenes=[]
for i,(a,b) in enumerate(scene_list):
    scenes.append({"scene_id":i+1,"start_seconds":a.get_seconds(),"end_seconds":b.get_seconds(),"duration":b.get_seconds()-a.get_seconds()})
if not scenes: scenes=[{"scene_id":1,"start_seconds":0.0,"end_seconds":duration,"duration":duration}]
# Candidate intro segment: first scene(s) until the first sustained scene change.
intro_end=scenes[0]["end_seconds"] if PRESERVE_OPENING_LOGO and scenes else 0.0
(OUT/"scene_manifest.json").write_text(json.dumps({"scenes":scenes,"opening_logo_candidate_end_seconds":intro_end,"opening_preserved":True},indent=2),encoding="utf-8")
print("Scenes:",len(scenes),"| opening/logo candidate:",round(intro_end,2),"seconds")
frames_dir=WORK/"sampled_frames"; frames_dir.mkdir(exist_ok=True)
step=max(1,int(round(fps/MAX_ANALYSIS_FPS)))
cap=cv2.VideoCapture(str(video_path)); rows=[]; idx=0
while True:
    ok,frame=cap.read()
    if not ok: break
    if idx%step==0:
        sec=idx/fps
        h,w=frame.shape[:2]; scale=min(1.0,MAX_LONG_SIDE/max(h,w))
        if scale<1: frame=cv2.resize(frame,(int(w*scale),int(h*scale)),interpolation=cv2.INTER_AREA)
        fp=frames_dir/f"frame_{idx:08d}.jpg"
        cv2.imwrite(str(fp),frame,[cv2.IMWRITE_JPEG_QUALITY,92])
        rows.append({"frame_index":idx,"timestamp_seconds":sec,"path":str(fp)})
    idx+=1
cap.release()
pd.DataFrame(rows).to_csv(OUT/"sampled_frames.csv",index=False)
print("Extracted sampled frames:",len(rows))

## 5. Character reference preparation — transparent plates and high-resolution exports
Each image is preserved as an immutable original. Background removal is attempted with rembg; a failure falls back to a clean resized RGB plate rather than stopping the entire run. Upscaling uses high-quality Lanczos (not AI hallucination). These are production-ready reference plates, not a rigged 3D character model.

In [ ]:
from rembg import remove, new_session
from PIL import Image
session=None
try: session=new_session("u2net")
except Exception as e: print("rembg model init warning:",repr(e))
character_manifest=[]
for i,src in enumerate(image_paths,1):
    im=Image.open(src).convert("RGB")
    original=ASSETS/f"character_{i}_original.png"; im.save(original)
    try:
        rgba=remove(im,session=session) if session else remove(im)
        if not isinstance(rgba,Image.Image): rgba=Image.open(__import__("io").BytesIO(rgba))
        rgba=rgba.convert("RGBA")
        seg_status="background_removed"
    except Exception as e:
        print(f"Character {i} segmentation fallback:",repr(e))
        rgba=im.convert("RGBA"); seg_status="rgb_fallback_no_segmentation"
    # upscale to a 2048px long edge without distorting aspect ratio
    long=max(rgba.size); scale=max(1.0,2048/long)
    target=(int(rgba.width*scale),int(rgba.height*scale))
    high=rgba.resize(target,Image.Resampling.LANCZOS)
    # Transparent PNG + neutral checker-free preview on white
    alpha_path=ASSETS/f"character_{i}_master_2048.png"; high.save(alpha_path,optimize=True)
    bg=Image.new("RGBA",high.size,(245,245,245,255)); bg.alpha_composite(high)
    preview=ASSETS/f"character_{i}_preview.jpg"; bg.convert("RGB").save(preview,quality=95)
    character_manifest.append({"character_id":f"CHAR_{i:02d}","source_file":str(src),"original_size":list(im.size),"master_size":list(high.size),"segmentation":seg_status,"master_png":str(alpha_path),"preview_jpg":str(preview),"rig_status":"not_rigged_requires_modeling_or_2D_rigging"})
(OUT/"character_manifest.json").write_text(json.dumps(character_manifest,indent=2),encoding="utf-8")
display(Image.open(ASSETS/"character_1_preview.jpg"))
display(Image.open(ASSETS/"character_2_preview.jpg"))

## 6. Pose/motion analysis
YOLO pose is used as a detector/tracker baseline. Only detections above a confidence threshold are exported. Missing detections remain null; the notebook never fills gaps with fabricated joint coordinates. Tracking IDs are provisional and should be reviewed in crowded/fast-action scenes.

In [ ]:
import torch
from ultralytics import YOLO
pose_model=YOLO("yolo11n-pose.pt")
motion_rows=[]
for rec in rows:
    result=pose_model.predict(source=rec["path"],conf=0.30,verbose=False,device=0 if torch.cuda.is_available() else "cpu")[0]
    if result.keypoints is None or result.boxes is None: continue
    kxy=result.keypoints.xy.cpu().numpy()
    kconf=result.keypoints.conf.cpu().numpy() if result.keypoints.conf is not None else np.zeros(kxy.shape[:2])
    boxes=result.boxes.xyxy.cpu().numpy()
    scores=result.boxes.conf.cpu().numpy()
    for j in range(len(kxy)):
        joints=[]
        for (x,y),c in zip(kxy[j],kconf[j]):
            joints.append({"x":float(x),"y":float(y),"confidence":float(c)} if float(c)>=0.25 else None)
        motion_rows.append({"frame_index":rec["frame_index"],"timestamp_seconds":rec["timestamp_seconds"],"detection_index":j,"bbox":[float(z) for z in boxes[j]],"detection_confidence":float(scores[j]),"keypoints":joints})
(OUT/"pose_detections.json").write_text(json.dumps({"model":"yolo11n-pose","confidence_threshold":0.30,"detections":motion_rows},separators=(",",":")),encoding="utf-8")
print("Pose detections:",len(motion_rows),"| sampled frames:",len(rows))
# Explicit quality summary
det_frames=len(set(x["frame_index"] for x in motion_rows))
quality={"sampled_frames":len(rows),"frames_with_pose":det_frames,"pose_frame_coverage":det_frames/max(1,len(rows)),"detections":len(motion_rows),"warning":"Pose detections are not guaranteed to identify the two protagonists; validate IDs and occlusions before animation."}
(OUT/"motion_quality.json").write_text(json.dumps(quality,indent=2),encoding="utf-8")
print(json.dumps(quality,indent=2))

## 7. Motion curves, camera proxies, and audio
Compute timestamp-aware center/scale changes for detections. Audio is extracted as a separate source asset and basic loudness envelope; advanced sound-event labeling can be added later.

In [ ]:
# Detection-level bbox motion proxies (not character identity guarantees)
byframe={}
for d in motion_rows:
    byframe.setdefault(d["frame_index"],[]).append(d)
curve_rows=[]
prev={}
for frame_idx,ds in sorted(byframe.items()):
    for j,d in enumerate(ds):
        x1,y1,x2,y2=d["bbox"]; cx=(x1+x2)/2; cy=(y1+y2)/2
        w=max(0,x2-x1); h=max(0,y2-y1); key=d["detection_index"]
        t=d["timestamp_seconds"]; old=prev.get(key)
        vx=vy=speed=None
        if old and t>old["t"]:
            vx=(cx-old["cx"])/(t-old["t"]); vy=(cy-old["cy"])/(t-old["t"]); speed=float((vx*vx+vy*vy)**0.5)
        curve_rows.append({"frame_index":frame_idx,"timestamp_seconds":t,"detection_index":key,"center_x":cx,"center_y":cy,"bbox_width":w,"bbox_height":h,"velocity_x":vx,"velocity_y":vy,"speed_px_s":speed})
        prev[key]={"t":t,"cx":cx,"cy":cy}
pd.DataFrame(curve_rows).to_csv(OUT/"motion_curves.csv",index=False)
audio_path=OUT/"source_audio.wav"
ar=run(["ffmpeg","-y","-i",str(video_path),"-vn","-ac","1","-ar","16000",str(audio_path)],check=False)
audio_status="extracted" if audio_path.exists() and audio_path.stat().st_size>1000 else "not_available"
(OUT/"audio_status.json").write_text(json.dumps({"status":audio_status,"ffmpeg_stderr":ar.stderr[-1000:]},indent=2),encoding="utf-8")
print("Audio:",audio_status)

## 8. Visual QA contact sheets
Create review sheets from sampled source frames and character masters. These allow quick visual checks before committing to a character rig or shot reconstruction.

In [ ]:
from PIL import Image, ImageDraw
def contact_sheet(paths,outfile,thumb=(256,144),cols=4):
    paths=list(paths); rows_n=max(1,(len(paths)+cols-1)//cols)
    sheet=Image.new("RGB",(cols*thumb[0],rows_n*(thumb[1]+24)),(28,28,32)); draw=ImageDraw.Draw(sheet)
    for i,p in enumerate(paths):
        im=Image.open(p).convert("RGB"); im.thumbnail(thumb)
        x=(i%cols)*thumb[0]; y=(i//cols)*(thumb[1]+24)
        sheet.paste(im,(x,y+24)); draw.text((x+5,y+5),Path(p).name[:36],fill="white")
    sheet.save(outfile,quality=92)
    return outfile
contact_sheet([r["path"] for r in rows[:80]],OUT/"source_contact_sheet.jpg")
contact_sheet(sorted(ASSETS.glob("*preview.jpg")),OUT/"character_contact_sheet.jpg",thumb=(512,512),cols=2)
display(Image.open(OUT/"character_contact_sheet.jpg"))
display(Image.open(OUT/"source_contact_sheet.jpg"))

## 9. Production handoff manifest
The deliverable contains source metadata, scene boundaries, character masters, pose detections, motion curves, audio, and QA images. Legacy analysis is optional and is never blindly merged with newly detected data.

In [ ]:
handoff={
 "project":PROJECT,"source_video":meta,"scene_manifest":"scene_manifest.json",
 "character_manifest":"character_manifest.json","pose_data":"pose_detections.json",
 "motion_curves":"motion_curves.csv","audio":"source_audio.wav" if audio_status=="extracted" else None,
 "qa":["source_contact_sheet.jpg","character_contact_sheet.jpg"],
 "limitations":[
  "Two still images do not define full 3D geometry, hidden surfaces, clothing back views, or a production rig.",
  "Pose tracks are detections, not guaranteed protagonist identities.",
  "The opening logo is preserved and reported separately; manually confirm the exact intro boundary.",
  "Final anime-grade animation requires approved character turnarounds, rigging, key animation, compositing, and shot-by-shot QA."
 ]
}
(OUT/"production_handoff.json").write_text(json.dumps(handoff,indent=2),encoding="utf-8")
# Inventory and checksums
inventory=[]
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        inventory.append({"path":str(p.relative_to(OUT)),"bytes":p.stat().st_size,"sha256":hashlib.sha256(p.read_bytes()).hexdigest()})
(OUT/"SHA256SUMS.json").write_text(json.dumps(inventory,indent=2),encoding="utf-8")
print("Output files:",len(inventory))

## 10. Package and publish to GitHub Releases
Add a GitHub Personal Access Token with **Contents: Read and write** to Colab Secrets as `GH_TOKEN`. The repository is public; the token is used only to create a Release and upload the generated artifact. It is never written into the notebook or output files.

In [ ]:
import zipfile, os, requests, getpass
zip_path=ROOT/"AL-DIDAH-production-artifacts.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in OUT.rglob("*"):
        if p.is_file(): z.write(p,Path("outputs")/p.relative_to(OUT))
print("Archive size MB:",round(zip_path.stat().st_size/1024/1024,2))

# Prefer Colab Secrets; fallback to hidden prompt.
gh_token=None
try:
    from google.colab import userdata
    gh_token=userdata.get("GH_TOKEN")
except Exception: pass
if not gh_token: gh_token=getpass.getpass("GitHub token (input hidden): ")
if not gh_token: raise RuntimeError("GH_TOKEN missing. Add it in Colab Secrets and rerun this publishing cell.")
headers={"Authorization":f"Bearer {gh_token}","Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28"}
api=f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}"
check=requests.get(api,headers=headers,timeout=30); check.raise_for_status()
# Reuse existing tag release if present; otherwise create it.
rel=requests.get(api+f"/releases/tags/{GITHUB_RELEASE_TAG}",headers=headers,timeout=30)
if rel.status_code==404:
    rel=requests.post(api+"/releases",headers=headers,json={"tag_name":GITHUB_RELEASE_TAG,"name":GITHUB_RELEASE_NAME,"body":"Automated AL-DIDAH analysis and production handoff artifacts. See production_handoff.json for scope and limitations.","draft":False,"prerelease":False},timeout=60)
rel.raise_for_status(); release=rel.json()
upload_url=release["upload_url"].split("{")[0]
with open(zip_path,"rb") as f:
    up=requests.post(upload_url+"?name="+zip_path.name,headers={**headers,"Content-Type":"application/zip"},data=f,timeout=600)
if up.status_code not in (201,422): up.raise_for_status()
print("Release:",release["html_url"])
print("Upload status:",up.status_code, up.json().get("browser_download_url","already uploaded / duplicate"))
(OUT/"github_release_receipt.json").write_text(json.dumps({"release_url":release["html_url"],"tag":GITHUB_RELEASE_TAG,"upload_status":up.status_code},indent=2),encoding="utf-8")


## 11. Final validation checklist
- [ ] Correct video was downloaded; metadata matches expected source.
- [ ] Opening logo/title segment is present in the contact sheet and not trimmed.
- [ ] Both character reference images match the intended designs.
- [ ] Transparent masters visually preserve all important details.
- [ ] Pose tracks are reviewed for identity swaps and false detections.
- [ ] Output ZIP exists and GitHub Release asset is downloadable.
- [ ] Production handoff limitations are understood; no false claim of a completed animated episode.